# Why bus stops are not a covariate: walking time to the nearest stop has no spread

**What this was for.** The proposal listed walking time to the nearest bus stop as a transit
covariate alongside MRT. Before spending ~45 minutes of OSRM time on the full 10,769 × 5,205
bus-stop matrix, this check asked whether the variable could carry any information at all.

**What it found.** Every HDB block sits within one to six minutes' walk of a bus stop (500-block
random sample, seed 4015, routed on the same OSRM foot network as the centre matrix: p10 1.3 min,
median 2.8, p90 4.7, max 8.9; 0% beyond 10 minutes), so the variable has essentially no variance
across estates and cannot explain differences in fees or supply. Walking time to the nearest MRT
exit ranges from under four minutes to over half an hour (median 9.0, p90 19.6; 44% beyond 10
minutes) and can. Straight-line distance for all blocks tells the same story: median 107 m to a
stop, 90th percentile 187 m, farthest block 424 m.

**Decision it informed** (`docs/decisions.md`, confirmed by the prof on 2026-10-30). Walking time to
the nearest bus stop is **not** computed for the full block set and is **not** a covariate in any
model; walking time to the nearest MRT exit **is** kept as an explanatory variable for fees (H-b)
and a transit-connectivity control. Neither is combined with the walk to a centre. If bus access
is ever revisited, the informative measure would be service frequency at the nearest stop (LTA
DataMall timetables), judged out of scope. The matrices themselves are built in
[`../1_Data_Preparation/1_06_build_walk_matrix.ipynb`](../1_Data_Preparation/1_06_build_walk_matrix.ipynb).
Original script: `scripts/check_bus_stop_variance.py` (Minh).

**Part 2 needs OSRM** on `http://localhost:5001`; when the server does not answer, that section
prints a note and keeps the numbers quoted above (the consultation brief quoted 2.7 / 9.2 min and
45% from an earlier draw of the same size; the difference is sampling noise).

**Inputs**
- `output/hdb_blocks.gpkg`, layer `hdb_points` (1_03); `data/BusStops_2026.geojson` (5,205 stops),
  `data/MRTStationExits_2026.geojson` (613 exits).

**Outputs**
- None (evidence printed in place).

In [1]:
import sys
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import pyogrio

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "bt4015").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from bt4015.paths import DATA, OUT, FIG, LARGE

from bt4015.osrm import CHUNK, DEST_SETS, lonlat, osrm_table, osrm_reachable

GPKG = OUT / "hdb_blocks.gpkg"
OSRM_BASE = "http://localhost:5001"
SAMPLE, SEED = 500, 4015     # blocks routed in part 2; the decision table used the same draw

## 1. Straight-line distance from every block to its nearest bus stop (no OSRM)

In [2]:
def q(x):
    """Distribution summary used for both parts: quantiles, max, CV, shares above 5 and 10."""
    x = pd.Series(x).dropna()
    return {"p10": x.quantile(.1), "median": x.median(), "p90": x.quantile(.9),
            "p99": x.quantile(.99), "max": x.max(), "CV": x.std() / x.mean(),
            ">5": (x > 5).mean() * 100, ">10": (x > 10).mean() * 100}


# 1. Blocks, stops and MRT exits in metres.
blocks = pyogrio.read_dataframe(GPKG, layer="hdb_points").to_crs(3414)
stops = pyogrio.read_dataframe(DEST_SETS["bus_stops"][0]).to_crs(3414)
mrt = pyogrio.read_dataframe(DEST_SETS["mrt_exits"][0]).to_crs(3414)
print(f"{len(blocks):,} blocks, {len(stops):,} bus stops, {len(mrt):,} MRT exits")

# 2. Nearest stop per block as the crow flies; one row per block even on ties.
sl = gpd.sjoin_nearest(blocks[["OBJECTID", "geometry"]], stops[["geometry"]], distance_col="d")
sl = sl.drop_duplicates("OBJECTID")
r = q(sl.d)
print(f"Straight-line to nearest bus stop, all {len(sl):,} blocks (m): "
      f"p10 {r['p10']:.0f} | median {r['median']:.0f} | p90 {r['p90']:.0f} | p99 {r['p99']:.0f} | "
      f"max {r['max']:.0f} | CV {r['CV']:.2f}")

10,769 blocks, 5,205 bus stops, 613 MRT exits
Straight-line to nearest bus stop, all 10,769 blocks (m): p10 51 | median 107 | p90 187 | p99 271 | max 424 | CV 0.47


## 2. Network walking time for a 500-block sample: bus stop vs MRT exit (needs OSRM)

The same `/table` calls as the matrix build, keeping only the minimum per origin. The MRT row is
the contrast: a transit variable that does vary across estates.

In [3]:
def nearest_min(base, o_xy, d_xy):
    """Minutes to the nearest destination for each origin, over CHUNK x CHUNK /table calls."""
    best = np.full(len(o_xy), np.inf)
    for i in range(0, len(o_xy), CHUNK):
        for j in range(0, len(d_xy), CHUNK):
            t = osrm_table(base, o_xy[i:i + CHUNK], d_xy[j:j + CHUNK], "duration") / 60.0
            best[i:i + CHUNK] = np.fmin(best[i:i + CHUNK], np.nanmin(np.where(np.isnan(t), np.inf, t), axis=1))
    return best


if not osrm_reachable(OSRM_BASE):
    print(f"OSRM not reachable on {OSRM_BASE}: part 2 skipped (figures from the 2026-10-30 run are in the title cell)")
else:
    # 1. Fixed random sample of blocks, routed to every stop and every exit.
    smp = blocks.sample(SAMPLE, random_state=SEED)
    o_xy = lonlat(smp)
    rows = []
    for name, dest in (("bus stop", stops), ("MRT exit", mrt)):
        rows.append({"nearest": name, **q(nearest_min(OSRM_BASE, o_xy, lonlat(dest)))})
    walk = pd.DataFrame(rows).set_index("nearest")
    print(f"Walking time (min) to nearest ..., {SAMPLE}-block random sample (seed {SEED}):")
    print(walk.round(1).to_string())

Walking time (min) to nearest ..., 500-block random sample (seed 4015):
          p10  median   p90   p99   max   CV    >5   >10
nearest                                                 
bus stop  1.3     2.8   4.7   6.6   8.9  0.5   7.0   0.0
MRT exit  3.7     9.0  19.6  31.5  37.8  0.6  81.0  44.2
